# Aula 6 - Imputação e Normalização de Dados

## Introdução

Nesse notebook, iremos responder alguns exercícios presentes na aula 06, "Imputação e Normalização de Dados". Essa atividade pode ser vista no tópico "Para casa" no notebook referente a essa aula. Seguiremos a lógica de blocos de códigos seguidos de blocos de Markdown para as interpretações.

Aproveitando, vamos importar algumas bibliotecas e a base de dados (`imoveis_sao_paulo.csv`) que será utilizada.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from sklearn.preprocessing import PowerTransformer

from google.colab import files

uploaded = files.upload()

imoveis = pd.read_csv("imoveis_sao_paulo.csv")

Saving imoveis_sao_paulo.csv to imoveis_sao_paulo.csv


## Preparação

Agora que carregamos a base, podemos padronizar os nomes das colunas numéricas e criar a variável categórica `property_type`, que identifica o imóvel como casa (`house`) ou apartamento (`apartment`) a partir do atributo `floor`. Logicamente, imóveis sem andar são casas.

In [2]:
imoveis = imoveis.rename(columns={
    "parking spaces": "parking_spaces",
    "hoa (R$)": "hoa",
    "rent amount (R$)": "rent_amount",
    "property tax (R$)": "property_tax",
    "total (R$)": "total",})

imoveis["floor"] = pd.to_numeric(imoveis["floor"], errors="coerce")
imoveis["property_type"] = np.where(imoveis["floor"].notna(), "apartment", "house")

print(imoveis["property_type"].value_counts(dropna=False))

property_type
apartment    25252
house         4454
Name: count, dtype: int64


## 1. Comparação entre o MAE da média e da mediana para `rooms`

Questão - "Compare o MAE da média e da mediana para rooms e relacione o resultado à assimetria e ao caráter discreto da variável."

Para avaliar qual das duas produz um menor erro para `rooms`, selecionamos os registros observados e mascaramos parte da amostra (20%) para simular valores ausentes. Assim, podemos comparar o MAE obtido ao preencher esses campos com a média e com a mediana calculadas no restante da base.

In [3]:
rooms_obs = imoveis["rooms"].dropna()

rng = np.random.default_rng(42)
mascara = rng.random(len(rooms_obs)) < 0.20

treino = rooms_obs[~mascara]
teste = rooms_obs[mascara]

media_rooms = treino.mean()
mediana_rooms = treino.median()

mae_media = np.abs(teste - media_rooms).mean()
mae_mediana = np.abs(teste - mediana_rooms).mean()

print(f"Assimetria (skew): {rooms_obs.skew():.3f}")
print(f"Média: {media_rooms:.2f} | MAE: {mae_media:.4f}")
print(f"Mediana: {mediana_rooms:.2f} | MAE: {mae_mediana:.4f}")

Assimetria (skew): 0.374
Média: 2.26 | MAE: 1.0043
Mediana: 2.00 | MAE: 0.9472


Observando os resultados, podemos perceber algumas coisas. Primeiramente, podemos ver que a mediana minimiza a soma dos desvios absolutos, fazendo dela uma boa opção para o MAE. Por outro lado, a média minimiza o erro quadrático. Considerando que `rooms` tem uma assimetria positiva (`0.374`), a cauda a direita, formada por imóveis maiores, puxa a média para cima (`2.26`), se distanciando da faixa onde a maioria dos imóveis se concentra.

Outro ponto interessante é sobre o caráter discreto da variável. O número de quartos só pode ser um valor inteiro, sendo assim, quando temos a mediana igual a `2.00`, acabamos acertando todos os imóveis mascarados que possuem dois quartos com erro zero. No entanto, quando temos uma média `2.26`, o mesmo não acontece. É impossível uma casa ter "dois quartos e meio", é um valor que não existe na prática e acaba gerando um erro diferente de zero em todas as entradas testadas, independente do número de quartos.

## 2. Imputação de aluguel por tier e zona

Questão - "Impute o aluguel por tier e zona. Depois, use o bairro como grupo e discuta o efeito de grupos pequenos."

Agora, vamos comparar duas estratégias de imputação para `rent_amount`. Primeiramente, vamos agrupar por `tier` e `zona`. Em seguida, fazemos o mesmo com `bairro`.

In [4]:
mediana_tier_zona = imoveis.groupby(["tier", "zona"])["rent_amount"].transform("median")
rent_imp_tier_zona = (
    imoveis["rent_amount"]
    .fillna(mediana_tier_zona)
    .fillna(imoveis["rent_amount"].median()))

mediana_bairro = imoveis.groupby("bairro")["rent_amount"].transform("median")
rent_imp_bairro = (
    imoveis["rent_amount"]
    .fillna(mediana_bairro)
    .fillna(imoveis["rent_amount"].median()))

contagem_bairros = imoveis.groupby("bairro")["rent_amount"].count()
print("Resumo do tamanho dos grupos por bairro:")
print(contagem_bairros.describe())

Resumo do tamanho dos grupos por bairro:
count     85.000000
mean     349.482353
std       19.567716
min      313.000000
25%      334.000000
50%      351.000000
75%      366.000000
max      394.000000
Name: rent_amount, dtype: float64


Analisando os valores, vemos que 85 bairros da base possuem entre 313 e 394 entradas válidas, com média de 349,5 imóveis por bairro.

Como essa base é bem povoada, o agrupamento apenas por `bairro` não sofre com falta de dados, mas perde a diferença de padrão do imóvel (`tier`). Isso acaba misturando imóveis econômicos e de alto padrão de um mesmo bairro.

Quando dividimos a base em subgrupos muito específicos, muitos desses grupos passam a ter pouquíssimas entradas. Um bairro com poucos imóveis, por exemplo. Isso é bem ruim, já que um único imóvel atípico pode mudar a mediana do grupo, além da possibilidade de grupos vazios.

## 3. Comparação de transformações para hoa entre apartamentos

Questão - "Entre apartamentos, compare log1p, Box-Cox e Yeo-Johnson para hoa. Qual produz menor assimetria? Qual você apresentaria a um gestor?"

Por fim, filtramos apenas os apartamentos com taxa de condomínio não nula (condição do Box-Cox) para comparar a redução de assimetria gerada por log1p, Box-Cox e Yeo-Johnson.

In [5]:
hoa_apt = imoveis.loc[
    (imoveis["property_type"] == "apartment") & (imoveis["hoa"] > 0),"hoa"]

hoa_log1p = np.log1p(hoa_apt)

pt_bc = PowerTransformer(method="box-cox", standardize=False)
hoa_bc = pt_bc.fit_transform(hoa_apt.to_frame()).ravel()

pt_yj = PowerTransformer(method="yeo-johnson", standardize=False)
hoa_yj = pt_yj.fit_transform(hoa_apt.to_frame()).ravel()

comparacao = pd.DataFrame({
    "Transformação": ["Original (hoa)", "log1p", "Box-Cox", "Yeo-Johnson"],
    "Lambda (λ)": [np.nan, 0.0, pt_bc.lambdas_[0], pt_yj.lambdas_[0]],
    "Assimetria (skew)": [
        hoa_apt.skew(),
        pd.Series(hoa_log1p).skew(),
        pd.Series(hoa_bc).skew(),
        pd.Series(hoa_yj).skew(),]})

print(comparacao.to_string(index=False))

 Transformação  Lambda (λ)  Assimetria (skew)
Original (hoa)         NaN           2.128776
         log1p    0.000000           0.149353
       Box-Cox   -0.085492           0.000406
   Yeo-Johnson   -0.086791           0.000435


Começando com uma assimetria acentuada a direita (`skew = 2.129`), os três métodos comprimem a cauda de forma considerável, mas ainda podemos ver algumas diferenças. Claramente, tanto Box-Cox (`skew = 0.000406`) quanto Yeo-Johnson (`skew = 0.000435`) geram, praticamente, uma simetria perfeita, superando o log1p (`skew = 0.149`). Isso ocorre porque ambos estimam o expoente lambda por máxima verossimilhança, aproximando a distribuição e a normalidade, enquanto o `log1p` aplica uma transformação fixa.

Antes de decidir qual eu apresentaria para um gestor, precisamos considerar algumas coisas. Não necessariamente a melhor simetria é a melhor escolha. Sobretudo, o ideal é sempre mostrar os valores na escala original em reais (R$), usando medianas e percentis. Se for necessário apresentar o dado transformado, a escolha é o log1p. Ele já corrige bem a assimetria e tem uma leitura simples. Box-Cox e o Yeo-Johnson trazem um ganho na simetria, mas dependem de um expoente quebrado (`$\lambda \approx -0,085$`) que não tem significado prático para quem toma decisões de negócio.